# Exemplo 11: Structured Streaming

**Objetivos:**
- Entender o conceito de stream processing
- Simular um stream a partir de ficheiros
- Calcular métricas em janelas temporais
- Escrever resultados para sink

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, count, avg, sum, max
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import time
import os

# Criar sessão Spark
spark = SparkSession.builder \
    .appName("Streaming_Demo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Definir Schema dos Dados

In [ ]:
# Schema para dados de trading
trade_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("exchange", StringType(), True)
])

print("Schema definido:")
for field in trade_schema.fields:
    print(f"  {field.name}: {field.dataType}")

## 2. Gerar Dados de Exemplo

Vamos criar dados simulados de transações (trades) em streaming.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Criar diretório para dados
input_dir = "stream_input"
os.makedirs(input_dir, exist_ok=True)

# Gerar dados simulados (divididos em batches)
np.random.seed(42)
symbols = ['BTC', 'ETH', 'SOL', 'ADA']
exchanges = ['Binance', 'Kraken', 'Coinbase']

base_time = datetime(2024, 1, 1, 9, 0, 0)

for batch in range(5):
    n_records = 100
    batch_time = base_time + timedelta(minutes=batch*5)
    
    data = {
        'timestamp': [batch_time + timedelta(seconds=int(x)) for x in np.random.randint(0, 300, n_records)],
        'symbol': np.random.choice(symbols, n_records),
        'price': np.random.uniform(100, 50000, n_records),
        'volume': np.random.uniform(0.1, 100, n_records),
        'exchange': np.random.choice(exchanges, n_records)
    }
    
    df_batch = pd.DataFrame(data)
    df_batch.to_json(f"{input_dir}/batch_{batch}.json", orient="records", lines=True)
    print(f"Criado batch_{batch}.json com {n_records} registos")

print(f"\nDados gerados em {input_dir}/")

## 3. Ler Stream de Dados

In [ ]:
# Criar stream a partir de ficheiros JSON
stream_df = spark.readStream \
    .schema(trade_schema) \
    .json(input_dir)

print("Stream criado!")
print(f"Is streaming: {stream_df.isStreaming}")
print("\nSchema:")
stream_df.printSchema()

## 4. Transformações em Streaming

In [ ]:
# Agregação simples: contagem por symbol
symbol_counts = stream_df.groupBy("symbol").count()

# Escrever para console (para debug)
query_console = symbol_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("symbol_counts") \
    .start()

print("Query iniciada. Aguardando dados...")
time.sleep(10)  # Deixar processar
query_console.stop()
print("Query parada.")

## 5. Agregações em Janelas Temporais

In [ ]:
# Tumbling Window: janelas de 5 minutos sem overlap
windowed_stats = stream_df \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window("timestamp", "5 minutes"),
        "symbol"
    ) \
    .agg(
        count("*").alias("n_trades"),
        avg("price").alias("avg_price"),
        sum("volume").alias("total_volume"),
        max("price").alias("max_price")
    )

# Mostrar resultado
query_window = windowed_stats.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

time.sleep(15)
query_window.stop()

## 6. Calcular VWAP (Volume Weighted Average Price)

Métrica comum em trading: preço médio ponderado pelo volume.

In [ ]:
from pyspark.sql.functions import expr

# Calcular VWAP por janela
vwap_calc = stream_df \
    .withColumn("price_volume", col("price") * col("volume")) \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window("timestamp", "5 minutes"),
        "symbol"
    ) \
    .agg(
        sum("price_volume").alias("sum_pv"),
        sum("volume").alias("sum_volume"),
        count("*").alias("count")
    ) \
    .withColumn("vwap", col("sum_pv") / col("sum_volume"))

# Mostrar VWAP
query_vwap = vwap_calc.select("window", "symbol", "vwap", "count").writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

time.sleep(15)
query_vwap.stop()

## 7. Output Modes

| Mode | Descrição | Use case |
|------|-----------|----------|
| **Append** | Apenas novas rows | Agregações com watermark |
| **Complete** | Tabela completa a cada trigger | Aggregações sem janela |
| **Update** | Rows modificadas desde último trigger | Map operations |

In [ ]:
# Escrever stream para Parquet (sink persistente)
output_dir = "stream_output"

query_file = windowed_stats.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", output_dir) \
    .option("checkpointLocation", "checkpoint") \
    .start()

print(f"Escrevendo para {output_dir}/")
time.sleep(20)
query_file.stop()
print("Stream para ficheiro parado.")

In [ ]:
# Ler os dados escritos
if os.path.exists(output_dir):
    output_df = spark.read.parquet(output_dir)
    print(f"Total de registos escritos: {output_df.count()}")
    output_df.show(10, truncate=False)
else:
    print("Diretório de saída não encontrado")

## 8. Ler de Kafka (exemplo de código)

```python
# Ler stream de Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "trades") \
    .load()

# Os dados Kafka têm schema fixo: key, value, topic, partition, offset, timestamp
# Normalmente value está em bytes e precisa de parse JSON
from pyspark.sql.functions import from_json, col

parsed = kafka_stream.select(
    from_json(col("value").cast("string"), trade_schema).alias("data")
).select("data.*")

# Escrever para Kafka
query_kafka = result_df.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "output") \
    .option("checkpointLocation", "checkpoint") \
    .start()
```

## Resumo

### Conceitos-chave:
1. **Stream = Tabela em contínua append**
2. **Watermark**: lidar com eventos atrasados
3. **Window functions**: agregações temporais
4. **Checkpoints**: tolerância a falhas

### Aplicações:
- Dashboards em tempo real
- Deteção de anomalias
- Métricas de trading (VWAP, volume)
- ETL contínuo

In [ ]:
# Limpar
import shutil
for d in [input_dir, output_dir, "checkpoint"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        
spark.stop()
print("Cleanup completo!")